<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 16. SimCLR — 대조학습 — A Simple Framework for Contrastive Learning of Visual Representations

- **원 논문:** [A Simple Framework for Contrastive Learning of Visual Representations](https://arxiv.org/abs/2002.05709)
- **저자 / 발표:** Ting Chen, Simon Kornblith, Mohammad Norouzi, Geoffrey Hinton · ICML (2020)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** cosine similarity, cross entropy, MLP, 데이터 증강

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

그림 2의 `x → 두 증강 view → encoder f → projection g → NT-Xent` 흐름을 작은 8×8 합성 패턴에 구현합니다. 양성 view의 cosine 유사도가 음성 쌍보다 커지는지, projection 이전 표현 `h`를 고정한 linear probe가 클래스를 분리하는지 확인합니다.

**원 논문과 다른 것**

ImageNet, ResNet-50, 큰 batch와 긴 학습은 생략하고 8×8 합성 영상·작은 MLP·짧은 full-batch 학습을 사용합니다. 따라서 ImageNet 정확도와 표 1~7의 수치를 재현하지 않습니다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2.1 및 그림 2 | 두 개의 확률적 view와 f/g 경로 | 두 view shape와 독립 잡음 |
| §2.1 식 (1) | cosine similarity | 정규화한 표현의 내적 |
| §2.1 식 (2), Algorithm 1 | 2N 표본의 NT-Xent | 대각선 제외·양성 index·유한 gradient |
| §2.2 및 그림 2 | projection 이전 h의 linear evaluation | 고정 encoder의 probe 정확도 |
| §3, 그림 4~5 | 증강과 temperature 미니 실험 | 양성/음성 유사도 차이 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\ell_{i,j}=-\log
\frac{\exp(\operatorname{sim}(z_i,z_j)/\tau)}
{\sum_{k\ne i}\exp(\operatorname{sim}(z_i,z_k)/\tau)}
$$

- $z_i,z_j$는 같은 image의 두 augmentation view, $\tau$는 temperature입니다.
- positive pair는 가깝게, batch의 다른 view는 멀게 만드는 NT-Xent objective입니다.
- augmentation, encoder, projection head, similarity matrix, masked loss를 단계별로 구현합니다.
- $2B$개 view의 embedding $[2B,D]$에서 similarity matrix $[2B,2B]$가 만들어집니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2.1 및 그림 2
- **노트북 구현:** 두 개의 확률적 view와 f/g 경로
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 두 view shape와 독립 잡음를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** ImageNet, ResNet-50, 큰 batch와 긴 학습은 생략하고 8×8 합성 영상·작은 MLP·짧은 full-batch 학습을 사용합니다. 따라서 ImageNet 정확도와 표 1~7의 수치를 재현하지 않습니다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(0)
print(ACCELERATOR.summary())
print("synthetic-data preprocessing stays on CPU; neural training uses DEVICE")

## 1) 한 샘플에서 두 개의 view 만들기

SimCLR의 label은 사람이 붙이지 않습니다. 같은 원본에서 독립적으로 뽑은 두 view가
양성 쌍이고, batch의 나머지가 음성입니다. 여기서는 crop·color distortion 대신
grayscale 합성 데이터에 맞춘 밝기 jitter와 Gaussian noise를 씁니다.

<details><summary>힌트</summary>
view마다 별도 random draw를 사용하고, 밝기는 sample별 `[N,1,1,1]`로 broadcast하세요.
</details>

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.1 및 그림 2
- **이 셀의 책임:** 두 개의 확률적 view와 f/g 경로
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 두 view shape와 독립 잡음. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 16-1: 독립적인 두 증강 view를 반환하세요.
def make_views(x: Tensor, noise: float = 0.08) -> tuple[Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def augment() -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


prototypes = torch.zeros(4, 1, 8, 8)
prototypes[0, :, :, 1:3] = 1
prototypes[1, :, :, 5:7] = 1
prototypes[2, :, 1:3, :] = 1
prototypes[3, :, 5:7, :] = 1
labels = torch.arange(4).repeat_interleave(24)
images = (prototypes[labels] + 0.04 * torch.randn(96, 1, 8, 8)).clamp(0, 1)
view_a, view_b = make_views(images)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 2) 식 (2): NT-Xent

`z=[z_i, z_j]`는 크기 `2N`입니다. 자기 자신은 denominator에서 제외하고, 앞쪽 view의
양성 index는 `i+N`, 뒤쪽은 `i-N`입니다. temperature `τ`는 logit의 선명도를 조절합니다.

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2.1 식 (1) / §2.1 식 (2), Algorithm 1
- **이 셀의 책임:** cosine similarity / 2N 표본의 NT-Xent
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 정규화한 표현의 내적 / 대각선 제외·양성 index·유한 gradient. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 16-2: 식 (1)의 cosine similarity와 식 (2)의 NT-Xent를 벡터화하세요.
def nt_xent(z1: Tensor, z2: Tensor, temperature: float = 0.2) -> Tensor:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §2.1 및 그림 2
- **이 셀의 책임:** 두 개의 확률적 view와 f/g 경로
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 두 view shape와 독립 잡음. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 16-3: encoder f와 비선형 projection head g를 별도 class로 구현하세요.
class TinyEncoder(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class ProjectionHead(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, hidden: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class TinySimCLR(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor]:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** §2.1 식 (2), Algorithm 1
- **이 셀의 책임:** 2N 표본의 NT-Xent
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 대각선 제외·양성 index·유한 gradient. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 16-4: 한 번의 contrastive update와 반복 학습을 구현하세요.
model = ACCELERATOR.move(TinySimCLR())
images_device, labels_device = ACCELERATOR.move(images, labels)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
losses = []


def contrastive_update(model, batch, optimizer, temperature=0.2):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2.2 및 그림 2
- **이 셀의 책임:** projection 이전 h의 linear evaluation
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 고정 encoder의 probe 정확도. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 16-5: encoder를 고정하고 h 위에 linear probe만 학습하세요.
for parameter in model.encoder.parameters():
    parameter.requires_grad_(False)
model.zero_grad(set_to_none=True)
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.